# PubMed Oncology MedGemma 27B Text SFT Fine-Tuning with Unsloth (4-bit QLoRA)

**Base Model:** MedGemma 27B Text IT (4-bit quantized) — Google's medical text LLM built on Gemma 3

**Dataset:** PubMed oncology datagen JSONL — multi-turn QA, continuation, treatment reasoning, beyond-evidence, and self-correction conversations across 11 cancer types

**Training Hardware:** NVIDIA DGX Spark (128GB unified memory)

**Inference Hardware:** NVIDIA A5000 24GB (~14GB weights + overhead fits comfortably)

**Chat Template:** Gemma 3 format — `<start_of_turn>role\ncontent<end_of_turn>`

**Architecture:** This is Phase 1 (SFT). Teaches the model clinical oncology reasoning. Phase 2 (DPO) refines response quality using preference pairs.

## How to run
**Press "Run All" and walk away.** Every cell is self-contained and runs in order.

## 1. Setup — Configuration, Environment, GPU Check

Installs missing packages, verifies GPU, sets all paths and hyperparameters. Safe to re-run.

In [1]:
import os, sys, subprocess, importlib, importlib.util

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CRITICAL: Disable Unsloth's FlexAttention for Gemma 3 on Blackwell         ║
# ║                                                                              ║
# ║  Unsloth's prefer_flex_attn_if_supported() forces flex_attention on Gemma 3, ║
# ║  overriding the default attention. FlexAttention's backward pass uses   ║
# ║  Triton kernels that exceed shared memory on sm_120 (Blackwell).             ║
# ║  Setting this to "0" prevents the override — Unsloth uses fast eager instead.           ║
# ║  All other torch.compile optimizations (fused LM head, fast LoRA, etc.)      ║
# ║  remain active.                                                              ║
# ║  MUST be set BEFORE `import unsloth`.                                        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
os.environ["UNSLOTH_ENABLE_FLEX_ATTENTION"] = "0"

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 1: INSTALL MISSING PACKAGES (safe to re-run, never clobbers NGC torch)║
# ║                                                                              ║
# ║  ORDER MATTERS:                                                              ║
# ║    1. torch check (no side effects)                                          ║
# ║    2. pip install small packages                                             ║
# ║    3. fix causal_conv1d (NGC ships broken build w/o CUDA extension)          ║
# ║    4. import unsloth FIRST (BEFORE transformers — required for patching)     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

def _pip(*args, env_extra=None):
    cmd = [sys.executable, "-m", "pip"] + list(args)
    env = os.environ.copy()
    if env_extra:
        env.update(env_extra)
    result = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if result.returncode != 0:
        print(f"  PIP FAILED: {' '.join(args)}")
        print(result.stderr[-500:] if result.stderr else result.stdout[-500:])
        return False
    return True

def _check_import(module_name):
    try:
        return importlib.import_module(module_name)
    except (ImportError, ModuleNotFoundError):
        return None

print("=" * 60)
print("ENVIRONMENT SETUP")
print("=" * 60)

# ── 1a. Verify NGC CUDA PyTorch is intact ──
import torch
if not torch.cuda.is_available():
    print("FATAL: torch.cuda.is_available() = False")
    print(f"  torch version: {torch.__version__}")
    if "+cpu" in torch.__version__ or "cpu" in torch.__version__:
        print("  NGC CUDA PyTorch was clobbered by pip. Recreate the container.")
    else:
        print("  GPU not passed through. Check Portainer: runtime=nvidia, NVIDIA_VISIBLE_DEVICES=all")
    raise RuntimeError("No GPU. Cannot continue. See messages above.")

print(f"  torch {torch.__version__} — CUDA {torch.version.cuda} — GPU: {torch.cuda.get_device_name(0)}")
print(f"  UNSLOTH_ENABLE_FLEX_ATTENTION = {os.environ.get('UNSLOTH_ENABLE_FLEX_ATTENTION')}")

# ── 1b. Small utility packages ──
for module, install_args in {
    "psutil":      ["install", "-q", "psutil"],
    "matplotlib":  ["install", "-q", "matplotlib"],
    "ipywidgets":  ["install", "-q", "ipywidgets"],
    "torchvision": ["install", "-q", "--no-deps", "torchvision"],
    "PIL":         ["install", "-q", "pillow"],
}.items():
    if _check_import(module) is None:
        print(f"  Installing {install_args[-1]}...")
        _pip(*install_args)

# ── 1c. Fix causal_conv1d ──
_causal_ok = False
_build_env = {
    "CAUSAL_CONV1D_FORCE_BUILD": "TRUE",
    "TORCH_CUDA_ARCH_LIST": "12.0;12.1",
}
try:
    from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn
    _causal_ok = True
    print("  causal_conv1d: OK (CUDA extension loaded)")
    for _k in list(sys.modules.keys()):
        if "causal_conv1d" in _k:
            del sys.modules[_k]
except (ImportError, ModuleNotFoundError, OSError):
    print("  causal_conv1d: CUDA extension missing — rebuilding from source (~3 min)...")
    _pip("uninstall", "-y", "causal-conv1d")
    _pip("cache", "remove", "causal_conv1d")
    for _k in list(sys.modules.keys()):
        if "causal_conv1d" in _k:
            del sys.modules[_k]
    importlib.invalidate_caches()
    ok = _pip("install", "--no-build-isolation", "--no-deps", "--force-reinstall",
              "--no-binary", "causal-conv1d", "causal-conv1d", env_extra=_build_env)
    if ok:
        importlib.invalidate_caches()
        try:
            from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn
            _causal_ok = True
            print("  causal_conv1d: rebuilt OK (CUDA extension working)")
            for _k in list(sys.modules.keys()):
                if "causal_conv1d" in _k:
                    del sys.modules[_k]
        except (ImportError, ModuleNotFoundError, OSError):
            print("  causal_conv1d: rebuild produced no CUDA ext — uninstalling for fallback")
            _pip("uninstall", "-y", "causal-conv1d")
            for _k in list(sys.modules.keys()):
                if "causal_conv1d" in _k:
                    del sys.modules[_k]
            importlib.invalidate_caches()
    else:
        print("  causal_conv1d: source build failed — uninstalling for fallback")
        _pip("uninstall", "-y", "causal-conv1d")
        importlib.invalidate_caches()

# ── 1d. Import unsloth FIRST, then transformers ──
# NOTE: UNSLOTH_ENABLE_FLEX_ATTENTION=0 is set above — unsloth reads it at import time.
for _k in list(sys.modules.keys()):
    if _k in ("transformers", "trl", "peft") or _k.startswith(("transformers.", "trl.", "peft.")):
        del sys.modules[_k]
importlib.invalidate_caches()

import unsloth
import transformers
print(f"  transformers {transformers.__version__}")

# ── 1e. Report final state ──
print()
for name, mod in [("unsloth", "unsloth"), ("transformers", "transformers"), ("trl", "trl"),
                   ("causal_conv1d", "causal_conv1d")]:
    m = _check_import(mod)
    v = getattr(m, "__version__", "installed") if m else "n/a"
    status = "OK" if m else "FALLBACK" if name == "causal_conv1d" else "MISSING"
    print(f"  {name:25s} {v:20s} [{status}]")

# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  STEP 2: CONFIGURATION                                                      ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

print()
print("=" * 60)
print("CONFIGURATION")
print("=" * 60)

# --- Paths (auto-detect Docker vs host) ---
if os.path.exists("/workspace/training/pubmed"):
    PROJECT_ROOT = "/workspace/training/pubmed"
    _env = "Docker (Unsloth container)"
elif os.path.exists("/workspace/pubmed"):
    PROJECT_ROOT = "/workspace/pubmed"
    _env = "Docker (legacy mount)"
else:
    PROJECT_ROOT = "/home/spark/projects/training/pubmed"
    _env = "Host (VS Code / venv)"

DATA_DIR    = f"{PROJECT_ROOT}/data"
OUTPUT_ROOT = f"{PROJECT_ROOT}/output"

# =========================== MODEL CONFIGURATION ===========================
BASE_LLM        = "unsloth/medgemma-27b-text-it-unsloth-bnb-4bit"
MODEL_NAME_BASE = "pubmed_oncologist_v2_medgemma_sft"

# =========================== INPUT DATA ===========================
INPUT_DATA_FILE = f"{DATA_DIR}/training-data/pubmed_oncologist_v2_tool_sft_messages.jsonl"

# =========================== OUTPUT DIRECTORIES ===========================
OUTPUT_BASE_DIR     = f"{OUTPUT_ROOT}/{MODEL_NAME_BASE}"
OUTPUT_DIR_ADAPTERS = f"{OUTPUT_BASE_DIR}/train"
LORA_OUTPUT_DIR     = f"{OUTPUT_BASE_DIR}/lora_adapters"

# =========================== TRAINING HYPERPARAMETERS ===========================
MAX_SEQ_LENGTH  = 4096
BATCH_SIZE      = 2        # smaller microbatch for 27B model (matches Qwen3 SFT)
GRAD_ACCUM      = 4        # effective batch = 2 * 4 = 8
LEARNING_RATE   = 2e-4
TARGET_EPOCHS   = 1

# =========================== LoRA CONFIGURATION ===========================
LORA_R              = 32
LORA_ALPHA          = 32
LORA_DROPOUT        = 0
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# =========================== INFERENCE TEST ===========================
TEST_PROMPT = "A 58-year-old woman with BRCA1-mutated high-grade serous ovarian cancer has progressed after platinum-based chemotherapy and a PARP inhibitor. What are the next treatment options?"

# --- Print summary ---
print(f"  Environment:  {_env}")
print(f"  PROJECT_ROOT: {PROJECT_ROOT}")
print(f"  Base model:   {BASE_LLM}")
print(f"  Model name:   {MODEL_NAME_BASE}")
print(f"  Input data:   {INPUT_DATA_FILE}")
print(f"  LoRA output:  {LORA_OUTPUT_DIR}")
print(f"  LoRA:         r={LORA_R}, alpha={LORA_ALPHA}, targets={len(LORA_TARGET_MODULES)} modules")
print(f"  Training:     batch={BATCH_SIZE} x grad_accum={GRAD_ACCUM} = effective {BATCH_SIZE*GRAD_ACCUM}")
print(f"  Precision:    4-bit QLoRA (pre-quantized NF4)")

# --- Verify paths ---
for path, label in [(INPUT_DATA_FILE, "Training data"), (PROJECT_ROOT, "Project root")]:
    exists = os.path.exists(path)
    print(f"  {'OK' if exists else 'MISSING':7s} {label}: {path}")
    if not exists:
        raise FileNotFoundError(f"{label} not found: {path}")

print()
print("Setup complete. All cells below can run sequentially.")

ENVIRONMENT SETUP
  torch 2.10.0a0+b558c986e8.nv25.11 — CUDA 13.0 — GPU: NVIDIA GB10
  UNSLOTH_ENABLE_FLEX_ATTENTION = 0
  causal_conv1d: OK (CUDA extension loaded)
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
  transformers 5.10.0.dev0

  unsloth                   2026.5.7             [OK]
  transformers              5.10.0.dev0          [OK]
  trl                       0.24.0               [OK]
  causal_conv1d             0.0.local            [OK]

CONFIGURATION
  Environment:  Docker (Unsloth container)
  PROJECT_ROOT: /workspace/training/pubmed
  Base model:   unsloth/medgemma-27b-text-it-unsloth-bnb-4bit
  Model name:   pubmed_oncologist_v2_medgemma_sft
  Input data:   /workspace/training/pubmed/data/training-data/pubmed_oncologist_v2_tool_sft_messages.jsonl
  LoRA output:  /workspace/training/pubmed/output/pubmed_oncologist_v2_medgemma_sft/lora_adapters
  LoRA:         r=32, alpha=32, tar

## 2. Load Dataset

Load the combined multi-turn ShareGPT JSONL from datagen.

- 33,349 conversations across 11 cancer types
- Data types: QA (with thinking), continuation, treatment reasoning, beyond-evidence, self-correction
- Standard ShareGPT format: `[system, human, gpt, human, gpt, ...]`
- System prompts are extracted from the JSONL at load time (stays in sync with datagen)

In [2]:
import json, os
from collections import defaultdict
from datasets import Dataset as HFDataset

print(f"LOADING TOOL-CALLING SFT DATA")
print(f"  File: {INPUT_DATA_FILE}")

raw_rows = []
cancer_type_counts = defaultdict(int)
source_counts = defaultdict(int)
system_prompts_by_cancer = {}  # cancer_type -> system prompt text

with open(INPUT_DATA_FILE) as f:
    for line in f:
        row = json.loads(line)

        cancer = row.get("cancer_type", "unknown")
        source = row.get("source", "unknown")
        cancer_type_counts[cancer] += 1
        source_counts[source] += 1

        # Keep messages format intact for tool-calling SFT.
        if "messages" in row:
            raw_rows.append({"messages": row["messages"]})
            # Extract system prompt if first message is system
            if row["messages"] and row["messages"][0].get("role") == "system":
                sys_msg = row["messages"][0].get("content", "")
                if sys_msg and cancer not in system_prompts_by_cancer:
                    system_prompts_by_cancer[cancer] = sys_msg
        elif "conversations" in row:
            raw_rows.append({"conversations": row["conversations"]})
            # Extract system prompt if first turn is system
            if row["conversations"] and row["conversations"][0].get("from") == "system":
                sys_msg = row["conversations"][0].get("value", "")
                if sys_msg and cancer not in system_prompts_by_cancer:
                    system_prompts_by_cancer[cancer] = sys_msg
        else:
            raise ValueError("Row missing both 'messages' and 'conversations'")

dataset = HFDataset.from_list(raw_rows)

print(f"\n{'='*50}")
print(f"Total dataset: {len(dataset)} conversations")
print(f"Cancer types: {len(cancer_type_counts)}")
print(f"Sources: {len(source_counts)}")
print(f"Unique system prompts: {len(system_prompts_by_cancer)}")
print(f"Columns: {dataset.column_names}")

print(f"\nPer-cancer breakdown:")
for ct, c in sorted(cancer_type_counts.items(), key=lambda x: -x[1]):
    print(f"  {ct:30s} {c:>5d} conversations")

print(f"\nPer-source breakdown:")
for src, c in sorted(source_counts.items(), key=lambda x: -x[1]):
    print(f"  {src:30s} {c:>5d} rows")


LOADING TOOL-CALLING SFT DATA
  File: /workspace/training/pubmed/data/training-data/pubmed_oncologist_v2_tool_sft_messages.jsonl

Total dataset: 9000 conversations
Cancer types: 5
Sources: 1
Unique system prompts: 5
Columns: ['messages']

Per-cancer breakdown:
  pubmed_ovarian_cancer           2177 conversations
  pubmed_breast_cancer            2072 conversations
  pubmed_colon_cancer             2071 conversations
  pubmed_prostate_cancer          1792 conversations
  pubmed_kidney_cancer             888 conversations

Per-source breakdown:
  tool_calling_augmentation       9000 rows


## 3. Validate & Summarize Dataset

Verify data quality: turn structure, non-empty responses, thinking tag presence.

In [3]:
from collections import Counter

bad_examples = []
empty_final_responses = []
tool_call_examples = 0

def _get_messages(example):
    if "messages" in example:
        return example["messages"]
    if "conversations" in example:
        # Fallback support for ShareGPT rows.
        role_map = {"system": "system", "human": "user", "gpt": "assistant"}
        return [{"role": role_map[t["from"]], "content": t["value"]} for t in example["conversations"]]
    raise ValueError("Example missing both messages and conversations")

for i, example in enumerate(dataset):
    msgs = _get_messages(example)

    if len(msgs) < 3:
        bad_examples.append((i, f"Expected >=3 messages, got {len(msgs)}"))
        continue

    # Must start with system, then user for this training format.
    if msgs[0].get("role") != "system" or msgs[1].get("role") != "user":
        bad_examples.append((i, "Expected first two roles to be system,user"))
        continue

    # Final assistant content must be non-empty.
    last = msgs[-1]
    if last.get("role") != "assistant" or not str(last.get("content", "")).strip():
        empty_final_responses.append(i)

    # Track explicit tool-calling examples.
    has_tool_call = any(m.get("role") == "assistant" and m.get("tool_calls") for m in msgs)
    if has_tool_call:
        tool_call_examples += 1

role_seq_dist = Counter(tuple(m.get("role") for m in _get_messages(ex)) for ex in dataset)

print("DATA QUALITY CHECK")
print(f"  Total examples: {len(dataset)}")
print(f"  Bad structure: {len(bad_examples)}")
print(f"  Empty final responses: {len(empty_final_responses)}")
print(f"  With explicit tool_calls: {tool_call_examples} ({100*tool_call_examples/len(dataset):.1f}%)")
print(f"  Without explicit tool_calls: {len(dataset)-tool_call_examples} ({100*(len(dataset)-tool_call_examples)/len(dataset):.1f}%)")

print(f"\nTop role sequences (up to 5):")
for seq, n in role_seq_dist.most_common(5):
    print(f"  {seq} -> {n}")

if bad_examples:
    print(f"\n  Bad examples (first 5):")
    for idx, reason in bad_examples[:5]:
        print(f"    Example {idx}: {reason}")

if empty_final_responses:
    print(f"\n  Filtering {len(empty_final_responses)} rows with empty/non-assistant final response...")
    bad_set = set(empty_final_responses)
    good_indices = [i for i in range(len(dataset)) if i not in bad_set]
    dataset = dataset.select(good_indices)
    print(f"  Dataset after filtering: {len(dataset)} examples")

print(f"\nCANCER TYPE DISTRIBUTION:")
max_name_len = max(len(n) for n in cancer_type_counts)
for name, count in sorted(cancer_type_counts.items(), key=lambda x: -x[1]):
    bar = "#" * max(1, count // 200)
    print(f"  {name:<{max_name_len}} {count:>5}  {bar}")
print(f"  {'TOTAL':<{max_name_len}} {sum(cancer_type_counts.values()):>5}")

print(f"\nDataset validated and ready for training")


DATA QUALITY CHECK
  Total examples: 9000
  Bad structure: 0
  Empty final responses: 0
  With explicit tool_calls: 9000 (100.0%)
  Without explicit tool_calls: 0 (0.0%)

Top role sequences (up to 5):
  ('system', 'user', 'assistant', 'tool', 'assistant') -> 9000

CANCER TYPE DISTRIBUTION:
  pubmed_ovarian_cancer   2177  ##########
  pubmed_breast_cancer    2072  ##########
  pubmed_colon_cancer     2071  ##########
  pubmed_prostate_cancer  1792  ########
  pubmed_kidney_cancer     888  ####
  TOTAL                   9000

Dataset validated and ready for training


## 4. Load Model & Tokenizer (4-bit)

In [4]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True
)

# ── CRITICAL: Override Unsloth's forced "eager" attention with Flash Attention 2 ──
# Unsloth forces attn_implementation="eager" for Gemma 3 (llama.py:2396 pops
# our kwarg, line 2427 passes "eager"). With "eager", Gemma3Attention.forward
# dispatches via ALL_ATTENTION_FUNCTIONS to eager_attention_forward which uses
# torch.matmul for Q@K^T — full O(n²) materialized attention. On a 27B model
# at seq_len 4096, this is the #1 reason training takes 182 hours instead of ~80.
#
# Gemma 3 uses the unified attention class (no separate SdpaAttention/FlashAttention
# subclasses). The dispatch is purely via config._attn_implementation at runtime.
# We override it AFTER Unsloth loads the model so flash_attention_forward runs instead.
#
# Requirements met:
#   - flash_attn 2.7.4 installed and working on sm_120 (Blackwell)
#   - GQA (32 query, 16 KV heads) — FA2 supports natively
#   - No softcap (attn_logit_softcapping=None in Gemma 3)
#   - sliding_window=1024 — FA2 handles via transformers wrapper
old_attn = getattr(model.config, "_attn_implementation", "unknown")
model.config._attn_implementation = "flash_attention_2"
print(f"  Attention override: {old_attn} -> flash_attention_2")

# Gemma tokenizer typically has pad_token already set (token id 0).
# Ensure it's configured; fall back to eos if missing.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    print(f"  Set pad_token = eos_token ({tokenizer.eos_token!r}, id={tokenizer.eos_token_id})")
else:
    print(f"  pad_token = {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")

model.config.pad_token_id = tokenizer.pad_token_id
if hasattr(model, "generation_config"):
    model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Model loaded: {BASE_LLM}")
print(f"  Precision: 4-bit QLoRA (pre-quantized NF4)")
print(f"  Max sequence length: {MAX_SEQ_LENGTH}")
print(f"  Vocab size: {len(tokenizer)}")
print(f"  GPU allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")

==((====))==  Unsloth 2026.5.7: Fast Gemma3 patching. Transformers: 5.10.0.dev0.
   \\   /|    NVIDIA GB10. Num GPUs = 1. Max memory: 121.689 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0a0+b558c986e8.nv25.11. CUDA: 12.1. CUDA Toolkit: 13.0. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+aa7bc36.d20260302. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/808 [00:00<?, ?it/s]

  Attention override: flash_attention_2 -> flash_attention_2
  pad_token = '<pad>' (id=0)
Model loaded: unsloth/medgemma-27b-text-it-unsloth-bnb-4bit
  Precision: 4-bit QLoRA (pre-quantized NF4)
  Max sequence length: 4096
  Vocab size: 262145
  GPU allocated: 16.6 GB


## 5. Format Dataset for Chat Template

Map ShareGPT roles to Gemma 3 chat template roles, apply the tokenizer's native chat template, then manual sequence packing for 100% token utilization.

**Gemma role mapping:** `system` → `system`, `human` → `user`, `gpt` → `model`

**Note:** The training data contains `<think>` reasoning blocks in the GPT responses. These are treated as regular content — MedGemma will learn to produce structured reasoning in this format during fine-tuning.


In [5]:
from datasets import Dataset as HFDataset
from tqdm.auto import tqdm
import json

# Fallback mapping if a non-tool ShareGPT row appears.
ROLE_MAP = {"system": "system", "human": "user", "gpt": "model"}

def _format_messages_for_gemma(messages):
    gemma_msgs = []
    # If the first message is system, keep its role as system
    has_system = messages and messages[0].get("role") == "system"
    if has_system:
        gemma_msgs.append({
            "role": "system",
            "content": messages[0].get("content", "").strip()
        })
        messages = messages[1:]
        
    for m in messages:
        role = m.get("role")
        content = m.get("content") or ""
        tool_calls = m.get("tool_calls")
        
        # Normalize role
        if role == "assistant":
            role = "model"
            
        if role == "user":
            gemma_msgs.append({"role": "user", "content": content})
        elif role == "model":
            if tool_calls:
                # Format tool call as clear custom XML tag text
                tc = tool_calls[0]
                fn = tc.get("function", {})
                args = fn.get("arguments", "{}")
                gemma_msgs.append({
                    "role": "model",
                    "content": f"<call:deep_research_pubmed>{args}</call:deep_research_pubmed>"
                })
            else:
                gemma_msgs.append({"role": "model", "content": content})
        elif role == "tool":
            # Map tool output as user inputs within structural XML tags
            gemma_msgs.append({
                "role": "user",
                "content": f"<response:deep_research_pubmed>\n{content}\n</response:deep_research_pubmed>"
            })
    return gemma_msgs

chat_conversations = []
for example in tqdm(dataset, desc="Formatting and Mapping roles"):
    if "messages" in example:
        messages = example["messages"]
    else:
        messages = [
            {"role": ROLE_MAP[turn["from"]], "content": turn["value"]}
            for turn in example["conversations"]
        ]
    chat_conversations.append(_format_messages_for_gemma(messages))

# Gemma 2 chat template
print("Applying chat template...")
formatted_texts = tokenizer.apply_chat_template(
    chat_conversations,
    tokenize=False,
)

# ── Manual sequence packing ──
eos_id = tokenizer.eos_token_id
num_conversations = 0
all_ids = []

for text in tqdm(formatted_texts, desc="Tokenizing & packing"):
    if not text:
        continue
    ids = tokenizer(text, add_special_tokens=False, truncation=False)["input_ids"]
    all_ids.extend(ids)
    all_ids.append(eos_id)
    num_conversations += 1

total_tokens = len(all_ids)
num_chunks = total_tokens // MAX_SEQ_LENGTH
all_ids = all_ids[:num_chunks * MAX_SEQ_LENGTH]
chunks = [all_ids[i * MAX_SEQ_LENGTH:(i + 1) * MAX_SEQ_LENGTH] for i in range(num_chunks)]

packed_texts = tokenizer.batch_decode(chunks, skip_special_tokens=False)
dataset = HFDataset.from_dict({"text": packed_texts})
dataset = dataset.shuffle(seed=42)

wasted = total_tokens - (num_chunks * MAX_SEQ_LENGTH)
print(f"Dataset packed: {num_conversations} conversations -> {num_chunks} chunks of {MAX_SEQ_LENGTH} tokens")
print(f"  Total tokens: {total_tokens:,}  |  Wasted (tail): {wasted:,} ({100*wasted/total_tokens:.1f}%)")
print(f"  Token utilization: ~100% (no padding)")
print(f"\n--- Sample packed text (first 500 chars) ---")
print(dataset[0]["text"][:500])


Formatting and Mapping roles:   0%|          | 0/9000 [00:00<?, ?it/s]

Applying chat template...


Tokenizing & packing:   0%|          | 0/9000 [00:00<?, ?it/s]

Dataset packed: 9000 conversations -> 7349 chunks of 4096 tokens
  Total tokens: 30,104,504  |  Wasted (tail): 3,000 (0.0%)
  Token utilization: ~100% (no padding)

--- Sample packed text (first 500 chars) ---
-pocket costs Prostate Cancer

**Training Snapshot Notice**: This tool result was synthesized from the existing validated PubMed oncology training corpus. It is for tool-calling alignment only and does not assert a real PMID unless one is shown explicitly.

--- **Article Details** ---
Search Query: How do Gleason score and comorbidities molecularly influence the mechanisms leading to higher out-of-pocket costs Prostate Cancer
Retrieved At (UTC): synthetic-training-snapshot
Total Articles Found: 


## 6. Add LoRA Adapters

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"LoRA adapters added (r={LORA_R}, alpha={LORA_ALPHA})")
print(f"  Trainable: {trainable:,} / {total:,} params ({100*trainable/total:.2f}%)")
print(f"  Target modules: {LORA_TARGET_MODULES}")

/usr/local/lib/python3.12/dist-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)


WARN  Python GIL is enabled: Multi-gpu quant acceleration for MoE models is sub-optimal and multi-core accelerated cpu packing is also disabled. We recommend Python >= 3.13.3t with Pytorch > 2.8 for mult-gpu quantization and multi-cpu packing with env `PYTHON_GIL=0`.


INFO  ENV: Auto setting CUDA_DEVICE_ORDER=PCI_BUS_ID for correctness.          


fatal: detected dubious ownership in repository at '/workspace/training/pubmed'
To add an exception for this directory, call:

	git config --global --add safe.directory /workspace/training/pubmed


INFO  

┌─────────────┐    ┌────────────────────────┐    ┌────────────┐    ┌─────────┐
│ GPT-QModel  │ -> │ ▓▓▓▓▓▓▓▓▓▓▓▓ 16bit     │ -> │ ▒▒▒▒ 8bit  │ -> │ ░░ 4bit │
└─────────────┘    └────────────────────────┘    └────────────┘    └─────────┘
GPT-QModel   : 7.0.0
Transformers : 5.10.0.dev0
Torch        : 2.10.0a0+b558c986e8.nv25.11
Triton       : 3.4.0+gitc5d671f9


LoRA adapters added (r=32, alpha=32)
  Trainable: 227,033,088 / 14,610,262,784 params (1.55%)
  Target modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']


## 7. Trainer Setup

In [7]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LENGTH,
        packing=False,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=5,
        num_train_epochs=TARGET_EPOCHS,
        learning_rate=LEARNING_RATE,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir=OUTPUT_DIR_ADAPTERS,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=3,
        report_to="none",
        dataset_num_proc=1,
    ),
)

effective_batch_size = BATCH_SIZE * GRAD_ACCUM
num_steps = (len(dataset) * TARGET_EPOCHS + effective_batch_size - 1) // effective_batch_size
print(f"Trainer configured (PubMed Oncology MedGemma 27B 4-bit QLoRA — pre-packed)")
print(f"  Effective batch size: {BATCH_SIZE} x {GRAD_ACCUM} = {effective_batch_size}")
print(f"  Epochs: {TARGET_EPOCHS}  |  Steps: ~{num_steps}")
print(f"  LR: {LEARNING_RATE}")
print(f"  Packing: manual (pre-packed, each example = {MAX_SEQ_LENGTH} tokens, zero padding)")
print(f"  Precision: {'bf16' if torch.cuda.is_bf16_supported() else 'fp16'}")
print(f"  Dataset: {len(dataset)} packed chunks")

Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/7349 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/multiprocess/popen_fork.py:66: DeprecationWarning: This process (pid=264) is multi-threaded, use of fork() may lead to deadlocks in the child.
  self.pid = os.fork()


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Trainer configured (PubMed Oncology MedGemma 27B 4-bit QLoRA — pre-packed)
  Effective batch size: 2 x 4 = 8
  Epochs: 1  |  Steps: ~919
  LR: 0.0002
  Packing: manual (pre-packed, each example = 4096 tokens, zero padding)
  Precision: bf16
  Dataset: 7349 packed chunks


## 8. Train

In [ ]:
from transformers.trainer_utils import get_last_checkpoint

last_ckpt = get_last_checkpoint(trainer.args.output_dir)
if last_ckpt is not None:
    print(f"Resuming from checkpoint: {last_ckpt}")
    result = trainer.train(resume_from_checkpoint=True)
else:
    print("No previous checkpoint found — starting fresh.")
    result = trainer.train()

print(f"\n✓ Training complete!")
print(f"  Final loss:     {result.training_loss:.4f}")
print(f"  Total steps:    {result.global_step}")
print(f"  Training time:  {result.metrics.get('train_runtime', 0) / 60:.1f} minutes")

No previous checkpoint found — starting fresh.


[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 7,349 | Num Epochs = 1 | Total steps = 919
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 227,033,088 of 27,236,035,328 (0.83% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,6.970207
2,6.991001
3,6.804852
4,6.092874
5,5.410761
6,5.016613
7,4.461318
8,4.296400
9,4.106110
10,3.888611


[transformers] Unsloth: Restored added_tokens_decoder metadata in /workspace/training/pubmed/output/pubmed_oncologist_v2_medgemma_sft/train/checkpoint-100/tokenizer_config.json.
[transformers] Unsloth: Preserved sentencepiece asset `tokenizer.model` in /workspace/training/pubmed/output/pubmed_oncologist_v2_medgemma_sft/train/checkpoint-100.
[transformers] Unsloth: Restored added_tokens_decoder metadata in /workspace/training/pubmed/output/pubmed_oncologist_v2_medgemma_sft/train/checkpoint-200/tokenizer_config.json.
[transformers] Unsloth: Preserved sentencepiece asset `tokenizer.model` in /workspace/training/pubmed/output/pubmed_oncologist_v2_medgemma_sft/train/checkpoint-200.
[transformers] Unsloth: Restored added_tokens_decoder metadata in /workspace/training/pubmed/output/pubmed_oncologist_v2_medgemma_sft/train/checkpoint-300/tokenizer_config.json.
[transformers] Unsloth: Preserved sentencepiece asset `tokenizer.model` in /workspace/training/pubmed/output/pubmed_oncologist_v2_medgem

## 9. Save LoRA Adapters

Save the trained LoRA adapters and system prompts. The DPO notebook (Phase 2) expects the LoRA at this path.

In [ ]:
import json
from pathlib import Path

Path(LORA_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"Saving LoRA adapters to {LORA_OUTPUT_DIR}...")
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)

# Save system prompts alongside adapters for inference use
prompts_path = f"{LORA_OUTPUT_DIR}/oncologist_system_prompts.json"
with open(prompts_path, "w") as f:
    json.dump(system_prompts_by_cancer, f, indent=2)

print(f"\nLoRA adapters saved!")
print(f"  Adapters:       {LORA_OUTPUT_DIR}")
print(f"  System prompts: {prompts_path} ({len(system_prompts_by_cancer)} cancer types)")
print(f"  DPO notebook expects LoRA at: {{PROJECT_ROOT}}/output/pubmed_oncologist_v2_sft/lora_adapters")

for p in sorted(Path(LORA_OUTPUT_DIR).iterdir()):
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"  {p.name:40s} {size_mb:>8.1f} MB")

## 10. Test Inference

Smoke test with oncology questions across different cancer types. The training data includes `<think>` reasoning blocks, so the model may produce them in responses.


In [ ]:
import warnings, logging, json
from transformers import TextStreamer

# Suppress transformers deprecation warning bug (msg % FutureWarning formatting error)
warnings.filterwarnings("ignore", category=FutureWarning)
logging.getLogger("transformers.modeling_attn_mask_utils").setLevel(logging.ERROR)

# Load model from saved adapters if not already in memory
if "model" not in dir() or model is None:
    from unsloth import FastLanguageModel
    print(f"Model not in memory — loading from saved adapters: {LORA_OUTPUT_DIR}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=LORA_OUTPUT_DIR,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
    )
    model.config._attn_implementation = "flash_attention_2"

if "system_prompts_by_cancer" not in dir() or not system_prompts_by_cancer:
    with open(f"{LORA_OUTPUT_DIR}/oncologist_system_prompts.json") as f:
        system_prompts_by_cancer = json.load(f)

FastLanguageModel.for_inference(model)

test_cancers = list(system_prompts_by_cancer.keys())[:3]

print(f"INFERENCE TEST — {len(test_cancers)} CANCER TYPES\n")

for cancer_type in test_cancers:
    system_prompt = system_prompts_by_cancer[cancer_type]

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": TEST_PROMPT},
    ]

    # Gemma 3 chat template — no enable_thinking parameter
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    print(f"{'='*60}")
    print(f"  CANCER TYPE: {cancer_type.upper()}")
    print(f"  Q: {TEST_PROMPT}")
    print(f"  A: ", end="")

    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        temperature=0.7,
        top_p=0.8,
        do_sample=True,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )
    print()

del inputs, outputs

## 11. Verify Adapter (Cold Reload)

Loads adapters from disk in a fresh model to confirm portability. Also verifies the DPO notebook can load these adapters.

In [ ]:
import gc, torch, json
from pathlib import Path

del model, tokenizer, trainer, dataset
gc.collect()
torch.cuda.empty_cache()

print("Cleared training model from memory")
print(f"  Loading adapter from: {LORA_OUTPUT_DIR}")

from unsloth import FastLanguageModel

model2, tokenizer2 = FastLanguageModel.from_pretrained(
    model_name=LORA_OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
model2.config._attn_implementation = "flash_attention_2"
FastLanguageModel.for_inference(model2)

with open(f"{LORA_OUTPUT_DIR}/oncologist_system_prompts.json") as f:
    reloaded_prompts = json.load(f)

test_cancer = list(reloaded_prompts.keys())[0]
test_system = reloaded_prompts[test_cancer]

messages = [
    {"role": "system", "content": test_system},
    {"role": "user", "content": TEST_PROMPT},
]

# Gemma 3 chat template
text = tokenizer2.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer2(text, return_tensors="pt").to(model2.device)

outputs = model2.generate(
    **inputs,
    max_new_tokens=2048,
    temperature=0.7,
    top_p=0.8,
    do_sample=True,
)

response = tokenizer2.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

print(f"\nADAPTER RELOAD TEST (cancer type: {test_cancer}):")
print(f"  Q: {TEST_PROMPT}")
print(f"  A: {response[:500]}")
print(f"\nAdapter loads cleanly from disk.")
print(f"DPO notebook (Phase 2) can now load this LoRA from: {LORA_OUTPUT_DIR}")

print(f"\nAdapter contents:")
for p in sorted(Path(LORA_OUTPUT_DIR).iterdir()):
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"  {p.name:40s} {size_mb:>8.1f} MB")

del model2, tokenizer2, inputs, outputs
gc.collect()
torch.cuda.empty_cache()